# Deepfake Face Detection — End-to-End Pipeline (Integrated)

> **Before running:** attach the two datasets (and the optional extra dataset) in the
> *Add Data* panel, then check `REAL_DIR` / `FAKE_DIR` / `EXTRA_DATASET` in the
> Configuration cell below match the mounted paths.

Outputs: trained model → `/kaggle/working/mobilenetv3_best.pth` · figures/results → `/kaggle/working/outputs` · cropped faces → `/kaggle/working/faces_cropped`

In [ ]:
# ============================================================
# Setup : Install retina-face (Kaggle: Settings -> Internet -> ON)
# ============================================================
import os as _os

# MUST be set before torch / tensorflow are imported.
# retina-face uses the TensorFlow backend, which by default grabs the ENTIRE
# GPU and never releases it, leaving PyTorch with crumbs (this caused CUDA OOM
# in Stage 2). These env vars cap TF's allocation and de-fragment PyTorch.
_os.environ["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"
_os.environ["TF_GPU_ALLOCATOR"] = "cuda_malloc_async"
_os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
_os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import subprocess, sys

result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "retina-face", "-q"],
    capture_output=True, text=True, timeout=120
)
if result.returncode == 0:
    print("retina-face installed successfully.")
else:
    print("retina-face not installed - face detection will fall back to center crop.")

## 1. Imports & Configuration

In [ ]:
# ============================================================
# Cell 1 : Imports & Configuration
# ============================================================

%matplotlib inline

import os
import io
import time
import random
from contextlib import contextmanager
from pathlib import Path

import numpy as np
import pandas as pd

from PIL import Image, ImageFilter

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report,
    f1_score, precision_score, recall_score, roc_auc_score,
)

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from torchvision import transforms
from torch.cuda.amp import autocast, GradScaler

from tqdm.auto import tqdm
import matplotlib.pyplot as plt
from IPython.display import display

# -----------------------------
# Reproducibility
# -----------------------------
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# -----------------------------
# Speed Optimizations
# -----------------------------
torch.backends.cudnn.benchmark = True

# -----------------------------
# Dataset Paths (update to match your Kaggle mounts)
# -----------------------------
REAL_DIR = Path(
    "/kaggle/input/datasets/philosopher0808/real-vs-ai-generated-faces-dataset/data_source/data_source/ffhq"
)

FAKE_DIR = Path(
    "/kaggle/input/datasets/mohannadaymansalah/stable-diffusion-dataaaaaaaaa"
)

# Optional extra dataset (Nano Banana 2.0) - skipped automatically if not mounted
EXTRA_DATASET = Path(
    "/kaggle/input/datasets/ahnuf05/ai-imagewith-nano-banana-2-0-vs-real-image/dataset"
)
USE_EXTRA_DATASET = True

# -----------------------------
# Output Paths (Kaggle working dir)
# -----------------------------
WORK_DIR = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path.cwd()
PREP_DIR = WORK_DIR / "faces_cropped"          # Vishaka: cached cropped faces
OUTPUT_DIR = WORK_DIR / "outputs"
MODEL_SAVE_PATH = WORK_DIR / "mobilenetv3_best.pth"

# -----------------------------
# Training Configuration
# -----------------------------
IMG_SIZE = 224

BATCH_SIZE = 64          # placeholder - re-verify with the Section 12 sweep below

NUM_WORKERS = 4

EPOCHS_STAGE1 = 3
EPOCHS_STAGE2 = 7

# -----------------------------
# Hyperparameters
# -----------------------------
# NOTE: these values were carried over from an earlier sweep run on a
# DIFFERENT (smaller / face-only) dataset. They are placeholders, not tuned
# for the current 3-dataset mix used in this notebook:
#   - philosopher0808/real-vs-ai-generated-faces-dataset (REAL_DIR)
#   - mohannadaymansalah/stable-diffusion-dataaaaaaaaa    (FAKE_DIR)
#   - ahnuf05/ai-imagewith-nano-banana-2-0-vs-real-image  (EXTRA_DATASET)
# Section 12 (Hyperparameter Sweep) now runs fresh against this exact data
# mix and auto-writes the winning config back into HPARAMS at runtime. Treat
# the values below as a starting point only until that sweep has run.
HPARAMS = {
    "learning_rate": 3e-4,
    "lr_stage2": 1e-5,
    "batch_size": 64,
    "weight_decay": 1e-4,
    "dropout": 0.2,
    "optimizer": "adamw",
    "scheduler": "cosine",
    "label_smoothing": 0.0,
}

# -----------------------------
# Vishaka preprocessing
# -----------------------------
DO_FACE_PREPROCESSING = True     # one-time RetinaFace crop (cached to disk)
FACE_PADDING = 0.20              # padding around detected face box
ROBUST_TRAIN_AUG = True          # JPEG/noise/blur robust augmentation pipeline
RUN_ROBUSTNESS_EVAL = True       # corruption robustness table (Vishaka CELL 7)
ROBUSTNESS_SUBSET = 300          # test images used for the robustness table

# -----------------------------
# Dataset Size (None -> use ALL images, e.g. 15000 -> 15K Real + 15K Fake)
# -----------------------------
IMAGES_PER_CLASS = 15000

# -----------------------------
# DataLoader
# -----------------------------
PIN_MEMORY = True
PERSISTENT_WORKERS = True

# -----------------------------
# Mixed Precision
# -----------------------------
scaler = GradScaler()

DEVICE = torch.device("cuda" if torch.cuda.is_available()
                      else "mps" if torch.backends.mps.is_available()
                      else "cpu")

USE_SCALER = DEVICE.type == "cuda"

print("=" * 60)
print("Device :", DEVICE)
if DEVICE.type == "cuda":
    print("GPU    :", torch.cuda.get_device_name(0))
print("=" * 60)

## 2. Build Dataset DataFrame

In [ ]:
# ============================================================
# Cell 2 : Build Dataset DataFrame  (Raunak)
# ============================================================

VALID_EXTENSIONS = (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp")


def get_image_paths(folder):
    """Recursively collect all image paths."""
    paths = []
    if not folder.is_dir():
        return paths
    for ext in VALID_EXTENSIONS:
        paths.extend(folder.rglob(f"*{ext}"))
        paths.extend(folder.rglob(f"*{ext.upper()}"))
    return sorted(paths)


def make_df(real_dir, fake_dir, domain="face_main"):
    """Build a filepath/label/domain DataFrame from real & fake image folders.

    `domain` tags where each row came from ("face_main" = core face dataset,
    "nano_banana" = cross-domain dataset) so later cells can report
    per-domain class balance and evaluate cross-domain accuracy separately
    without re-scanning folders.
    """
    real = get_image_paths(Path(real_dir))
    fake = get_image_paths(Path(fake_dir))
    real_df = pd.DataFrame({"filepath": [str(x) for x in real], "label": 0, "domain": domain})
    fake_df = pd.DataFrame({"filepath": [str(x) for x in fake], "label": 1, "domain": domain})
    return pd.concat([real_df, fake_df], ignore_index=True)


def check_split_balance(train_df, val_df, test_df):
    """Print per-split class balance and per-domain class balance.

    Targets the two most common causes of a large train/test accuracy gap:
    (1) a class ratio that differs sharply between splits, and (2) a domain
    (e.g. nano_banana) that is disproportionately represented in one split.
    """
    print("\n" + "=" * 60)
    print("Stratification / Class-Balance Check")
    print("=" * 60)
    for name, d in [("Train", train_df), ("Validation", val_df), ("Test", test_df)]:
        counts = d["label"].value_counts().sort_index()
        total = len(d)
        real_pct = counts.get(0, 0) / total * 100 if total else 0
        fake_pct = counts.get(1, 0) / total * 100 if total else 0
        print(f"{name:<12} Real: {counts.get(0, 0):>6,} ({real_pct:5.1f}%)  "
              f"Fake: {counts.get(1, 0):>6,} ({fake_pct:5.1f}%)  Total: {total:,}")

    print("\nPer-Domain Breakdown")
    print("-" * 60)
    for name, d in [("Train", train_df), ("Validation", val_df), ("Test", test_df)]:
        if "domain" not in d.columns:
            continue
        dom_counts = d.groupby(["domain", "label"]).size().unstack(fill_value=0)
        print(f"\n{name}:")
        print(dom_counts.to_string())

    train_fake_ratio = (train_df["label"] == 1).mean()
    test_fake_ratio = (test_df["label"] == 1).mean()
    gap = abs(train_fake_ratio - test_fake_ratio)
    if gap > 0.05:
        print(f"\n[WARNING] Train fake-ratio ({train_fake_ratio:.2%}) and "
              f"Test fake-ratio ({test_fake_ratio:.2%}) differ by {gap:.2%}. "
              "This alone can explain part of a train/test accuracy gap.")
    else:
        print(f"\n[OK] Train/Test class ratio gap is small ({gap:.2%}).")


def check_split_leakage(train_df, val_df, test_df):
    """Verify no exact filepath is shared across splits (data leakage check).

    If the same source image appears in both train and test, the model can
    effectively memorize it during training while the score on the rest of
    test stays low - a direct contributor to a train/test accuracy gap.
    """
    print("\n" + "=" * 60)
    print("Split Leakage Check (duplicate filepaths across splits)")
    print("=" * 60)
    train_set = set(train_df["filepath"])
    val_set = set(val_df["filepath"])
    test_set = set(test_df["filepath"])

    train_val = train_set & val_set
    train_test = train_set & test_set
    val_test = val_set & test_set

    if train_val or train_test or val_test:
        print(f"[WARNING] Leakage detected -> "
              f"train∩val: {len(train_val)}, train∩test: {len(train_test)}, "
              f"val∩test: {len(val_test)}")
    else:
        print("[OK] No filepath overlap between train/val/test splits.")


def build_splits(real_dir, fake_dir, extra_dataset=None, images_per_class=IMAGES_PER_CLASS):
    """Load image paths, optionally sample per class, split 80/10/10 and
    optionally merge the extra dataset into the splits."""
    print("Loading image paths...")
    real_images = get_image_paths(real_dir)
    fake_images = get_image_paths(fake_dir)

    print(f"Found {len(real_images):,} Real images")
    print(f"Found {len(fake_images):,} Fake images")

    if images_per_class is not None:
        random.shuffle(real_images)
        random.shuffle(fake_images)
        real_images = real_images[:min(images_per_class, len(real_images))]
        fake_images = fake_images[:min(images_per_class, len(fake_images))]

    df = pd.concat([
        pd.DataFrame({"filepath": [str(p) for p in real_images], "label": 0, "domain": "face_main"}),
        pd.DataFrame({"filepath": [str(p) for p in fake_images], "label": 1, "domain": "face_main"}),
    ], ignore_index=True).sample(frac=1, random_state=SEED).reset_index(drop=True)

    print("\nDataset Summary")
    print(df["label"].value_counts().to_string())
    print(f"\nTotal Images : {len(df):,}")

    train_df, temp_df = train_test_split(
        df, test_size=0.20, stratify=df["label"], random_state=SEED
    )
    val_df, test_df = train_test_split(
        temp_df, test_size=0.50, stratify=temp_df["label"], random_state=SEED
    )

    # Optional extra dataset (Nano Banana 2.0, cross-domain) - merged per
    # split like MAIN so the model sees some cross-domain examples during
    # training. Every row keeps domain="nano_banana" so Priority 4
    # (Cross-Domain Testing) can filter test_df down to only this domain and
    # report accuracy on it separately from face_main.
    if extra_dataset is not None and extra_dataset.is_dir() and USE_EXTRA_DATASET:
        print("\nExtra Dataset detected, merging...")
        extra_train_df = make_df(extra_dataset / "train" / "real",
                                 extra_dataset / "train" / "fake", domain="nano_banana")
        extra_val_df = make_df(extra_dataset / "val" / "real",
                               extra_dataset / "val" / "fake", domain="nano_banana")
        extra_test_df = make_df(extra_dataset / "test" / "real",
                                extra_dataset / "test" / "fake", domain="nano_banana")
        train_df = pd.concat([train_df, extra_train_df], ignore_index=True)
        val_df = pd.concat([val_df, extra_val_df], ignore_index=True)
        test_df = pd.concat([test_df, extra_test_df], ignore_index=True)

    train_df = train_df.sample(frac=1, random_state=SEED).reset_index(drop=True)
    val_df = val_df.sample(frac=1, random_state=SEED).reset_index(drop=True)
    test_df = test_df.sample(frac=1, random_state=SEED).reset_index(drop=True)

    print("\n" + "=" * 50)
    print("Final Dataset")
    print("=" * 50)
    print(f"Train      : {len(train_df):,}")
    print(f"Validation : {len(val_df):,}")
    print(f"Test       : {len(test_df):,}")

    check_split_balance(train_df, val_df, test_df)
    check_split_leakage(train_df, val_df, test_df)

    return train_df, val_df, test_df


# ---- Execute: load paths, sample, split + merge extra dataset ----
train_df, val_df, test_df = build_splits(
    REAL_DIR, FAKE_DIR,
    extra_dataset=EXTRA_DATASET if USE_EXTRA_DATASET else None,
)

display(train_df.head())

## 3. Face Detection & Alignment Preprocessing  *(Vishaka)*

Runs **RetinaFace** on every image **once** and caches the cropped face to
`/kaggle/working/faces_cropped`. Already-processed images are skipped, so this
step is safe to interrupt and resume. Training then loads pre-cropped faces.

In [ ]:
# ============================================================
# Cell 3 : Face Detection & Alignment Preprocessing  (Vishaka)
# ============================================================

try:
    from retinaface import RetinaFace
    RETINA_AVAILABLE = True
except ImportError:
    RETINA_AVAILABLE = False
    print("WARNING: retina-face not installed - face detection will fall back "
          "to center crop. Enable internet in Kaggle and restart.")

try:
    import cv2
    CV2_AVAILABLE = True
except ImportError:
    CV2_AVAILABLE = False


def center_crop(image_pil):
    """Center-crop to square then resize to IMG_SIZE."""
    w, h = image_pil.size
    crop_size = min(w, h)
    left = (w - crop_size) // 2
    top = (h - crop_size) // 2
    return image_pil.crop((left, top, left + crop_size, top + crop_size)).resize(
        (IMG_SIZE, IMG_SIZE), Image.Resampling.LANCZOS
    )


def crop_and_align_face(image_pil, padding=FACE_PADDING, debug=False):
    """
    Detect the largest face with RetinaFace, add padding, crop and resize to
    IMG_SIZE x IMG_SIZE. Falls back to center crop if unavailable / no face.
    """
    if RETINA_AVAILABLE:
        try:
            if CV2_AVAILABLE:
                img_bgr = cv2.cvtColor(np.array(image_pil), cv2.COLOR_RGB2BGR)
                faces = RetinaFace.detect_faces(img_bgr)
            else:
                faces = RetinaFace.detect_faces(np.array(image_pil))

            if isinstance(faces, dict) and len(faces) > 0:
                largest_face = max(
                    faces.values(),
                    key=lambda f: (
                        (f["facial_area"][2] - f["facial_area"][0]) *
                        (f["facial_area"][3] - f["facial_area"][1])
                    )
                )
                x1, y1, x2, y2 = largest_face["facial_area"]
                face_w = x2 - x1
                face_h = y2 - y1
                x1 = max(0, int(x1 - padding * face_w))
                y1 = max(0, int(y1 - padding * face_h))
                x2 = min(image_pil.width, int(x2 + padding * face_w))
                y2 = min(image_pil.height, int(y2 + padding * face_h))
                return image_pil.crop((x1, y1, x2, y2)).resize(
                    (IMG_SIZE, IMG_SIZE), Image.Resampling.LANCZOS
                )
        except Exception as e:
            if debug:
                print(f"RetinaFace Error: {e}")

    return center_crop(image_pil)


# Registry shared across train/val/test calls: out_path -> (source_filepath, split_name)
# Lets us detect two real bugs that both contribute to a train/test accuracy
# gap: (a) a cache-key COLLISION - two different source images hashing to the
# same output filename (would silently corrupt one of them), and (b) the SAME
# source image appearing in more than one split (leakage introduced through
# the extra-dataset merge, not through this caching step itself).
_face_cache_registry = {}


def preprocess_and_save(df, split_name, prep_dir):
    """
    ONE-TIME face preprocessing: run RetinaFace on every image once and save
    the cropped face to disk. Already-processed images are skipped, so this is
    safe to interrupt and resume.
    """
    new_paths = []
    skipped = 0
    collisions = 0
    cross_split_dupes = 0

    for _, row in tqdm(df.iterrows(), total=len(df), desc=f"Preprocessing {split_name}"):
        src_path = row["filepath"]
        rel = os.path.relpath(src_path, "/").replace(os.sep, "_")
        out_path = os.path.join(prep_dir, rel + ".jpg")

        if out_path in _face_cache_registry:
            prev_src, prev_split = _face_cache_registry[out_path]
            if prev_src != src_path:
                collisions += 1
            elif prev_split != split_name:
                cross_split_dupes += 1
        else:
            _face_cache_registry[out_path] = (src_path, split_name)

        if not os.path.exists(out_path):
            try:
                img = Image.open(src_path).convert("RGB")
                face = crop_and_align_face(img)
                face.save(out_path, "JPEG", quality=95)
            except Exception:
                # Last resort: use the original file untouched
                out_path = src_path
                skipped += 1
        new_paths.append(out_path)

    print(f"{split_name}: {len(new_paths)} processed, {skipped} fallbacks.")
    if collisions:
        print(f"  [WARNING] {collisions} cache-key collisions - two different "
              f"source images mapped to the same output file. One overwrote "
              f"the other; investigate the affected filenames.")
    if cross_split_dupes:
        print(f"  [WARNING] {cross_split_dupes} images in this split were "
              f"already cached under a DIFFERENT split - the same source "
              f"file exists in more than one split (leakage). Check "
              f"REAL_DIR/FAKE_DIR/EXTRA_DATASET for duplicate files.")
    if not collisions and not cross_split_dupes:
        print("  [OK] No cache-key collisions or cross-split duplicates detected.")

    return new_paths


def run_face_preprocessing(train_df, val_df, test_df, prep_dir):
    """Apply cached RetinaFace cropping to every split. (Vishaka)"""
    print("\n" + "=" * 60)
    print("Face Preprocessing (RetinaFace, one-time, cached)")
    print("=" * 60)
    os.makedirs(prep_dir, exist_ok=True)
    train_df["filepath"] = preprocess_and_save(train_df, "train", prep_dir)
    val_df["filepath"] = preprocess_and_save(val_df, "val", prep_dir)
    test_df["filepath"] = preprocess_and_save(test_df, "test", prep_dir)
    print("Preprocessing complete. Training will now load pre-cropped faces.")
    return train_df, val_df, test_df


def show_preprocessing_samples(df, n=4):
    """Visualize a few cropped faces so you can eyeball preprocessing
    quality right after it runs (Real + Fake, one row each)."""
    fig, axes = plt.subplots(2, n, figsize=(3.2 * n, 7))
    for row_idx, label in enumerate([0, 1]):
        subset = df[df["label"] == label]
        samples = subset.sample(min(n, len(subset)), random_state=SEED)
        for col_idx, (_, row) in enumerate(samples.iterrows()):
            ax = axes[row_idx, col_idx]
            try:
                img = Image.open(row["filepath"]).convert("RGB")
                ax.imshow(img)
            except Exception as e:
                ax.text(0.5, 0.5, f"load error\n{e}", ha="center", va="center")
            ax.set_title(("Real" if label == 0 else "Fake") +
                        (f" [{row['domain']}]" if "domain" in row else ""),
                        fontsize=10)
            ax.axis("off")
        for col_idx in range(len(samples), n):
            axes[row_idx, col_idx].axis("off")
    fig.suptitle("Preprocessed (cropped) Face Samples", fontsize=13)
    plt.tight_layout()
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    fig.savefig(OUTPUT_DIR / "preprocessing_samples.png", dpi=150)
    plt.show()
    plt.close(fig)


# ---- Execute: one-time RetinaFace cropping (cached, resumable) ----
if DO_FACE_PREPROCESSING:
    train_df, val_df, test_df = run_face_preprocessing(train_df, val_df, test_df, PREP_DIR)
    show_preprocessing_samples(train_df, n=4)

In [ ]:
# ---- Release TensorFlow GPU memory held by the RetinaFace backend ----
# so that PyTorch gets the full GPU back for training.
try:
    import gc
    import tensorflow as tf

    tf.keras.backend.clear_session()
    del tf

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    print("Freed TensorFlow GPU memory (RetinaFace backend).")

except Exception as e:
    print(f"Cleanup skipped: {e}")

## 4. Robust Transformation Pipeline  *(Vishaka)*

In [ ]:
# ============================================================
# Cell 4 : Robust Transforms  (Vishaka)
# ============================================================

class JPEGCompression(object):
    """Simulate forensic JPEG re-compression with random quality."""

    def __init__(self, quality_range=(30, 90)):
        self.quality_range = quality_range

    def __call__(self, img):
        quality = int(np.random.randint(self.quality_range[0],
                                        self.quality_range[1] + 1))
        output = io.BytesIO()
        img.save(output, format="JPEG", quality=quality)
        output.seek(0)
        return Image.open(output).convert("RGB")


class AddGaussianNoise(object):
    """Add Gaussian noise to a normalized-free tensor (before Normalize)."""

    def __init__(self, mean=0.0, std=0.02):
        self.mean = mean
        self.std = std

    def __call__(self, tensor):
        noise = torch.randn(tensor.size()) * self.std + self.mean
        return torch.clamp(tensor + noise, 0.0, 1.0)


def build_train_transform():
    """Robust augmentation pipeline (Vishaka) used when ROBUST_TRAIN_AUG."""
    return transforms.Compose([
        JPEGCompression(quality_range=(30, 90)),
        transforms.RandomResizedCrop(IMG_SIZE, scale=(0.85, 1.0)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.ColorJitter(brightness=0.25, contrast=0.25,
                               saturation=0.25, hue=0.05),
        transforms.GaussianBlur(kernel_size=(5, 5), sigma=(0.1, 2.0)),
        transforms.ToTensor(),
        AddGaussianNoise(mean=0.0, std=0.02),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225]),
    ])


def build_val_transform():
    return transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225]),
    ])


def _unnormalize(tensor):
    """Undo the ImageNet Normalize() for display purposes."""
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    return torch.clamp(tensor * std + mean, 0, 1)


def show_augmentation_samples(df, n_variants=5):
    """Apply the train-time augmentation pipeline to one source image
    several times so you can see what the model actually trains on."""
    row = df.sample(1, random_state=SEED).iloc[0]
    base_img = Image.open(row["filepath"]).convert("RGB")

    fig, axes = plt.subplots(1, n_variants + 1, figsize=(3.2 * (n_variants + 1), 4))
    axes[0].imshow(base_img.resize((IMG_SIZE, IMG_SIZE)))
    axes[0].set_title("Original")
    axes[0].axis("off")

    for i in range(1, n_variants + 1):
        augmented = train_transform(base_img)
        display_img = _unnormalize(augmented).permute(1, 2, 0).numpy()
        axes[i].imshow(display_img)
        axes[i].set_title(f"Augmented #{i}")
        axes[i].axis("off")

    fig.suptitle(f"Train Augmentation Samples - {Path(row['filepath']).name} "
                f"({'Real' if row['label'] == 0 else 'Fake'})", fontsize=12)
    plt.tight_layout()
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    fig.savefig(OUTPUT_DIR / "augmentation_samples.png", dpi=150)
    plt.show()
    plt.close(fig)


# ---- Execute: build transforms ----
train_transform = build_train_transform() if ROBUST_TRAIN_AUG else build_val_transform()
val_transform = build_val_transform()

show_augmentation_samples(train_df, n_variants=5)

## 5. Dataset & DataLoaders  

In [ ]:
# ============================================================
# Cell 5 : Dataset & DataLoaders
# ============================================================

class PreprocessedDataset(Dataset):
    """Loads pre-cropped face images from disk. No face detection at train time."""

    def __init__(self, dataframe, transform=None):
        self.df = dataframe
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = self.df.iloc[idx]["filepath"]
        label = int(self.df.iloc[idx]["label"])
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, label


def build_dataloaders(train_df, val_df, test_df, batch_size=BATCH_SIZE,
                      num_workers=NUM_WORKERS, robust_aug=ROBUST_TRAIN_AUG):
    train_transform = build_train_transform() if robust_aug else build_val_transform()
    val_transform = build_val_transform()

    train_dataset = PreprocessedDataset(train_df, transform=train_transform)
    val_dataset = PreprocessedDataset(val_df, transform=val_transform)
    test_dataset = PreprocessedDataset(test_df, transform=val_transform)

    loader_kwargs = dict(
        num_workers=num_workers,
        pin_memory=PIN_MEMORY,
        persistent_workers=PERSISTENT_WORKERS and num_workers > 0,
    )

    train_loader = DataLoader(train_dataset, batch_size=batch_size,
                              shuffle=True, **loader_kwargs)
    val_loader = DataLoader(val_dataset, batch_size=batch_size,
                            shuffle=False, **loader_kwargs)
    test_loader = DataLoader(test_dataset, batch_size=batch_size,
                             shuffle=False, **loader_kwargs)

    print("=" * 60)
    print("DataLoader Ready")
    print("=" * 60)
    print(f"Train Images : {len(train_dataset):,}")
    print(f"Val Images   : {len(val_dataset):,}")
    print(f"Test Images  : {len(test_dataset):,}")

    return train_loader, val_loader, test_loader


# ---- Execute: build loaders ----
train_loader, val_loader, test_loader = build_dataloaders(
    train_df, val_df, test_df,
    batch_size=BATCH_SIZE,
    robust_aug=ROBUST_TRAIN_AUG,
)

# Quick sanity check
images, labels = next(iter(train_loader))
print("\nBatch Shape :", images.shape)
print("Labels Shape:", labels.shape)
print("Tensor Type :", images.dtype)

## 6. Model — MobileNetV3-Large 

In [ ]:
# ============================================================
# Cell 6 : Model - MobileNetV3-Large
# ============================================================

def build_model(dropout=HPARAMS["dropout"], num_classes=2):
    """MobileNetV3-Large with frozen backbone and a configurable-dropout head."""
    from torchvision.models import mobilenet_v3_large, MobileNet_V3_Large_Weights

    model = mobilenet_v3_large(weights=MobileNet_V3_Large_Weights.DEFAULT)

    # Freeze entire backbone
    for param in model.features.parameters():
        param.requires_grad = False

    # Replace classification head (dropout configurable - Rohit)
    in_features = model.classifier[0].in_features
    model.classifier = nn.Sequential(
        nn.Linear(in_features, 1280),
        nn.Hardswish(),
        nn.Dropout(p=dropout),
        nn.Linear(1280, num_classes),
    )
    return model


def build_optimizer(model, opt_type, lr, weight_decay, stage):
    """Adam or AdamW on the given parameter set (Rohit)."""
    if stage == 1:
        params = model.classifier.parameters()
    else:
        params = filter(lambda p: p.requires_grad, model.parameters())

    if opt_type == "adam":
        return optim.Adam(params, lr=lr, weight_decay=weight_decay)
    return optim.AdamW(params, lr=lr, weight_decay=weight_decay)


def build_scheduler(optimizer, sched_type, epochs):
    """CosineAnnealingLR or ReduceLROnPlateau (Rohit)."""
    if sched_type == "plateau":
        return optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode="min", factor=0.5, patience=2
        )
    return optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=epochs, eta_min=1e-7
    )


@contextmanager
def amp_context():
    if DEVICE.type in ("cuda", "mps"):
        with autocast():
            yield
    else:
        yield


def compute_metrics(labels, preds, probs):
    metrics = {
        "acc": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, zero_division=0),
        "precision": precision_score(labels, preds, zero_division=0),
        "recall": recall_score(labels, preds, zero_division=0),
    }
    metrics["auc"] = (roc_auc_score(labels, probs)
                      if len(np.unique(labels)) > 1 else 0.0)
    return metrics


# ---- Execute: build model ----
model = build_model(dropout=HPARAMS["dropout"]).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters()
                       if p.requires_grad)

print("=" * 60)
print("Model Ready")
print("=" * 60)
print(f"Total Parameters     : {total_params:,}")
print(f"Trainable Parameters : {trainable_params:,}")

## 7. Training Functions  

In [ ]:
# ============================================================
# Cell 7 : Training Functions
# ============================================================

def train_one_epoch(model, loader, optimizer, criterion):
    """One training epoch with mixed precision. (Raunak)"""
    model.train()
    running_loss = 0.0
    running_correct = 0
    total = 0

    pbar = tqdm(loader, leave=False)
    for images, labels in pbar:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        optimizer.zero_grad()

        with amp_context():
            outputs = model(images)
            loss = criterion(outputs, labels)

        if USE_SCALER:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            optimizer.step()

        preds = outputs.argmax(dim=1)
        running_loss += loss.item() * images.size(0)
        running_correct += (preds == labels).sum().item()
        total += labels.size(0)

        pbar.set_postfix(loss=f"{loss.item():.4f}")

    return running_loss / total, running_correct / total


def validate(model, loader, criterion):
    """Validation loop returning loss, accuracy and full metrics (Rohit)."""
    model.eval()
    running_loss = 0.0
    total = 0
    all_labels, all_preds, all_probs = [], [], []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)

            with amp_context():
                outputs = model(images)
                loss = criterion(outputs, labels)

            probs = torch.softmax(outputs, dim=1)[:, 1]
            preds = outputs.argmax(dim=1)

            running_loss += loss.item() * images.size(0)
            total += labels.size(0)
            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    return running_loss / total, compute_metrics(all_labels, all_preds, all_probs)


def train_stage(model, loaders, optimizer, scheduler, criterion, epochs,
                stage_name, sched_type):
    """Train one stage (classifier head or fine-tune) with best-model saving."""
    train_loader, val_loader, _ = loaders
    best_val_acc = 0.0

    print("=" * 60)
    print(f"{stage_name}")
    print("=" * 60)

    for epoch in range(epochs):
        start = time.time()

        train_loss, train_acc = train_one_epoch(
            model, train_loader, optimizer, criterion
        )
        val_loss, val_metrics = validate(model, val_loader, criterion)

        if sched_type == "plateau":
            scheduler.step(val_loss)
        else:
            scheduler.step()

        val_acc = val_metrics["acc"]
        elapsed = time.time() - start

        print(
            f"Epoch [{epoch+1}/{epochs}] "
            f"| Train Loss {train_loss:.4f} "
            f"| Train Acc {train_acc*100:.2f}% "
            f"| Val Loss {val_loss:.4f} "
            f"| Val Acc {val_acc*100:.2f}% "
            f"| Val AUC {val_metrics['auc']:.4f} "
            f"| Time {elapsed:.1f}s"
        )

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(
                {
                    "epoch": epoch + 1,
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "val_acc": val_acc,
                },
                MODEL_SAVE_PATH,
            )
            print(f"Best model saved ({best_val_acc*100:.2f}%)")

    print(f"\n{stage_name} Complete - Best Val Acc: {best_val_acc*100:.2f}%")
    return best_val_acc

## 8. Stage 1 & Stage 2 Training  *(Raunak, using Rohit's tuned hparams)*

- **Stage 1** – train only the classification head (backbone frozen)
- **Stage 2** – fine-tune the last 25% of the backbone at a lower LR

In [ ]:
# ============================================================
# Cell 8 : Stage 1 & Stage 2 Training
# ============================================================

criterion = nn.CrossEntropyLoss(label_smoothing=HPARAMS["label_smoothing"])

# ---------------- Stage 1 : train classifier head ----------------
optimizer = build_optimizer(
    model, HPARAMS["optimizer"], HPARAMS["learning_rate"],
    HPARAMS["weight_decay"], stage=1,
)
scheduler = build_scheduler(optimizer, HPARAMS["scheduler"], EPOCHS_STAGE1)

best_val_acc = train_stage(
    model, (train_loader, val_loader, test_loader),
    optimizer, scheduler, criterion, EPOCHS_STAGE1,
    "Stage 1 : Training Classifier", HPARAMS["scheduler"],
)

# ---------------- Stage 2 : fine-tune last 25% of backbone ----------------
checkpoint = torch.load(MODEL_SAVE_PATH, map_location=DEVICE)
model.load_state_dict(checkpoint["model_state_dict"])
best_val_acc = max(best_val_acc, checkpoint["val_acc"])

# Free cached CUDA memory before unfreezing the backbone (Stage 2 needs
# to retain activations for backward through the unfrozen blocks).
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

feature_blocks = list(model.features.children())
num_blocks = len(feature_blocks)
start_block = int(num_blocks * 0.75)

print(f"\nTotal Feature Blocks : {num_blocks}")
print(f"Unfreezing Blocks    : {start_block} -> {num_blocks-1}")

for block in feature_blocks[start_block:]:
    for param in block.parameters():
        param.requires_grad = True

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable Parameters : {trainable:,}")

optimizer = build_optimizer(
    model, HPARAMS["optimizer"], HPARAMS["lr_stage2"],
    HPARAMS["weight_decay"], stage=2,
)
scheduler = build_scheduler(optimizer, HPARAMS["scheduler"], EPOCHS_STAGE2)

best_val_acc = train_stage(
    model, (train_loader, val_loader, test_loader),
    optimizer, scheduler, criterion, EPOCHS_STAGE2,
    "Stage 2 : Fine-Tuning", HPARAMS["scheduler"],
)

print("\n")
print("=" * 60)
print("Training Finished")
print("=" * 60)
print(f"Best Validation Accuracy : {best_val_acc*100:.2f}%")

## 9. Final Evaluation

In [ ]:
# ============================================================
# Cell 9 : Final Evaluation
# ============================================================

def evaluate(model, test_loader, criterion, save_fig=True):
    """Test-set evaluation: loss/acc, classification report, confusion matrix."""
    model.eval()
    test_loss = 0.0
    total = 0
    all_labels, all_preds, all_probs = [], [], []

    with torch.no_grad():
        pbar = tqdm(test_loader)
        for images, labels in pbar:
            images = images.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)

            with amp_context():
                outputs = model(images)
                loss = criterion(outputs, labels)

            probs = torch.softmax(outputs, dim=1)[:, 1]
            preds = outputs.argmax(dim=1)

            test_loss += loss.item() * images.size(0)
            total += labels.size(0)
            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    test_loss /= total
    test_metrics = compute_metrics(all_labels, all_preds, all_probs)

    print("\n" + "=" * 60)
    print("Test Results")
    print("=" * 60)
    print(f"Test Loss     : {test_loss:.4f}")
    print(f"Test Accuracy : {test_metrics['acc']*100:.2f}%")
    print(f"Test F1       : {test_metrics['f1']:.4f}")
    print(f"Test AUC      : {test_metrics['auc']:.4f}")

    print("\nClassification Report\n")
    print(classification_report(all_labels, all_preds,
                                target_names=["Real", "Fake"], digits=4))

    cm = confusion_matrix(all_labels, all_preds)
    print("\nConfusion Matrix")
    print(cm)

    if save_fig:
        os.makedirs(OUTPUT_DIR, exist_ok=True)
        plt.figure(figsize=(6, 6))
        plt.imshow(cm)
        plt.title("Confusion Matrix")
        plt.colorbar()
        plt.xticks([0, 1], ["Real", "Fake"])
        plt.yticks([0, 1], ["Real", "Fake"])
        for i in range(2):
            for j in range(2):
                plt.text(j, i, cm[i, j], ha="center", va="center", fontsize=14)
        plt.xlabel("Predicted")
        plt.ylabel("True")
        plt.tight_layout()
        plt.savefig(OUTPUT_DIR / "confusion_matrix.png", dpi=150)
        plt.show()
        plt.close()

    return test_loss, test_metrics


# ---- Execute: load best model and evaluate on test set ----
model.load_state_dict(torch.load(MODEL_SAVE_PATH, map_location=DEVICE)[
    "model_state_dict"])
model.eval()

test_loss, test_metrics = evaluate(model, test_loader, criterion)

## 9b. Augmentation Ablation Training - No-Augmentation Model  *(Vishakha)*

Priority 1 asks to literally compare a model **trained with** augmentation
against one **trained without** it, on the same train/val/test split. The
main model above (Sections 6-9) is the "with augmentation" model. This
section trains a second MobileNetV3-Large from scratch with
`ROBUST_TRAIN_AUG` forced off - same data, same architecture, same
`HPARAMS`, same epoch counts - and saves it to a separate checkpoint
(`mobilenetv3_noaug.pth`) so both models can be compared on the same
held-out test set here, and later on your own uploaded images.

> `TRAIN_NOAUG_MODEL = True` by default - this runs a full second Stage 1 +
> Stage 2 training pass, roughly doubling total training time. Set to
> `False` to skip.

In [ ]:
# ============================================================
# Cell 9b : Augmentation Ablation Training (No-Aug Model)  (Vishakha)
# ============================================================

MODEL_SAVE_PATH_NOAUG = WORK_DIR / "mobilenetv3_noaug.pth"
TRAIN_NOAUG_MODEL = True   # set False to skip the no-augmentation comparison model


def train_full_model(robust_aug, save_path, run_label):
    """Train a fresh MobileNetV3-Large end-to-end (Stage 1 + Stage 2) with
    the given augmentation setting, saving best checkpoints to `save_path`.
    Reuses build_dataloaders/build_model/build_optimizer/build_scheduler/
    train_stage exactly as the main pipeline does, just pointed at a
    separate checkpoint file so it never overwrites the main model."""
    print("\n" + "#" * 60)
    print(f"# {run_label}  (ROBUST_TRAIN_AUG={robust_aug})")
    print("#" * 60)

    t_loader, v_loader, e_loader = build_dataloaders(
        train_df, val_df, test_df,
        batch_size=HPARAMS["batch_size"],
        robust_aug=robust_aug,
    )

    m = build_model(dropout=HPARAMS["dropout"]).to(DEVICE)
    crit = nn.CrossEntropyLoss(label_smoothing=HPARAMS["label_smoothing"])

    global MODEL_SAVE_PATH
    original_save_path = MODEL_SAVE_PATH
    MODEL_SAVE_PATH = save_path  # train_stage() saves best checkpoints to this global path
    try:
        opt = build_optimizer(m, HPARAMS["optimizer"], HPARAMS["learning_rate"],
                              HPARAMS["weight_decay"], stage=1)
        sch = build_scheduler(opt, HPARAMS["scheduler"], EPOCHS_STAGE1)
        best_val_acc = train_stage(
            m, (t_loader, v_loader, e_loader), opt, sch, crit, EPOCHS_STAGE1,
            f"{run_label} - Stage 1", HPARAMS["scheduler"],
        )

        ckpt = torch.load(save_path, map_location=DEVICE)
        m.load_state_dict(ckpt["model_state_dict"])
        best_val_acc = max(best_val_acc, ckpt["val_acc"])

        blocks = list(m.features.children())
        start = int(len(blocks) * 0.75)
        for b in blocks[start:]:
            for prm in b.parameters():
                prm.requires_grad = True

        opt = build_optimizer(m, HPARAMS["optimizer"], HPARAMS["lr_stage2"],
                              HPARAMS["weight_decay"], stage=2)
        sch = build_scheduler(opt, HPARAMS["scheduler"], EPOCHS_STAGE2)
        best_val_acc = train_stage(
            m, (t_loader, v_loader, e_loader), opt, sch, crit, EPOCHS_STAGE2,
            f"{run_label} - Stage 2", HPARAMS["scheduler"],
        )
    finally:
        MODEL_SAVE_PATH = original_save_path

    m.load_state_dict(torch.load(save_path, map_location=DEVICE)["model_state_dict"])
    m.eval()
    print(f"\n{run_label} Complete - Best Val Acc: {best_val_acc*100:.2f}%")
    return m


def compare_augmentation(model_with_aug, model_without_aug, test_loader, criterion):
    """Head-to-head test-set comparison: with vs without augmentation. (Vishakha)"""
    print("\n" + "=" * 60)
    print("Augmentation Ablation - Test Set Comparison (Priority 1)")
    print("=" * 60)
    _, metrics_with = validate(model_with_aug, test_loader, criterion)
    _, metrics_without = validate(model_without_aug, test_loader, criterion)

    table = pd.DataFrame([
        {"Model": "With Augmentation", "Accuracy (%)": round(metrics_with["acc"] * 100, 2),
         "F1": round(metrics_with["f1"], 4), "AUC": round(metrics_with["auc"], 4)},
        {"Model": "Without Augmentation", "Accuracy (%)": round(metrics_without["acc"] * 100, 2),
         "F1": round(metrics_without["f1"], 4), "AUC": round(metrics_without["auc"], 4)},
    ])
    print(table.to_string(index=False))
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    table.to_csv(OUTPUT_DIR / "augmentation_ablation_results.csv", index=False)
    return table


# ---- Execute: train the no-augmentation comparison model ----
model_noaug = None
if TRAIN_NOAUG_MODEL:
    model_noaug = train_full_model(robust_aug=False, save_path=MODEL_SAVE_PATH_NOAUG,
                                   run_label="No-Augmentation Model")
    augmentation_ablation_table = compare_augmentation(model, model_noaug, test_loader, criterion)
else:
    print("TRAIN_NOAUG_MODEL is False - skipping no-augmentation comparison model.")

## 10. Grad-CAM Explainability  *(Somendu)*

In [ ]:
# ============================================================
# Cell 10 : Grad-CAM
# ============================================================

import torch.nn.functional as F


class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.activations = None
        self.gradients = None
        self.forward_handle = target_layer.register_forward_hook(
            self._save_activations
        )

    def _save_activations(self, module, inputs, output):
        self.activations = output
        output.register_hook(self._save_gradients)

    def _save_gradients(self, gradients):
        self.gradients = gradients

    def __call__(self, input_tensor, class_idx=None):
        was_training = self.model.training
        self.model.eval()
        self.model.zero_grad(set_to_none=True)
        input_tensor = input_tensor.requires_grad_(True)

        logits = self.model(input_tensor)
        probabilities = torch.softmax(logits, dim=1).detach().cpu()

        if class_idx is None:
            class_idx = logits.argmax(dim=1).item()

        logits[0, class_idx].backward()
        weights = self.gradients.mean(dim=(2, 3), keepdim=True)
        heatmap = (weights * self.activations).sum(dim=1, keepdim=True)
        heatmap = torch.relu(heatmap)
        heatmap = F.interpolate(
            heatmap,
            size=input_tensor.shape[-2:],
            mode="bilinear",
            align_corners=False,
        )[0, 0]

        heatmap = heatmap - heatmap.min()
        heatmap = heatmap / (heatmap.max() + 1e-8)

        if was_training:
            self.model.train()

        return heatmap.detach().cpu().numpy(), int(class_idx), probabilities

    def close(self):
        self.forward_handle.remove()


def _predict_test_subset(model, df, transform, max_n=300):
    """Batch-predict a random subset of df so we can separate correct vs
    incorrect predictions without running Grad-CAM (which needs a full
    backward pass) over the entire test set."""
    subset = df.sample(min(max_n, len(df)), random_state=SEED).reset_index(drop=True)
    loader = DataLoader(
        PreprocessedDataset(subset, transform=transform),
        batch_size=BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
    )
    model.eval()
    preds = []
    with torch.no_grad():
        for images, _ in loader:
            images = images.to(DEVICE, non_blocking=True)
            with amp_context():
                out = model(images)
            preds.extend(out.argmax(dim=1).cpu().numpy().tolist())
    subset = subset.copy()
    subset["pred"] = preds
    subset["correct"] = subset["pred"] == subset["label"]
    return subset


def run_gradcam_examples(model, test_df, transform, n_per_group=2, save_fig=True):
    """Explain n CORRECT + n INCORRECT predictions per class with Grad-CAM.
    (Raunak / Somendu)

    Priority 5 explicitly asks for correct-prediction and incorrect-
    prediction examples as two distinct groups, not just n samples per true
    label - this scores predictions first, then samples from each of the
    four (true label x correct/incorrect) buckets.
    """
    classes = ["Real", "Fake"]
    scored = _predict_test_subset(model, test_df, transform, max_n=300)
    gradcam = GradCAM(model, model.features[-1])

    def make_gradcam_overlay(image, heatmap, alpha=0.45):
        image = image.convert("RGB").resize((IMG_SIZE, IMG_SIZE))
        heatmap_rgb = (plt.get_cmap("jet")(heatmap)[..., :3] * 255).astype(np.uint8)
        heatmap_image = Image.fromarray(heatmap_rgb).resize(image.size)
        overlay = Image.blend(image, heatmap_image, alpha=alpha)
        return image, heatmap_image, overlay

    os.makedirs(OUTPUT_DIR, exist_ok=True)

    for true_label in [0, 1]:
        for is_correct in [True, False]:
            group = scored[(scored["label"] == true_label) & (scored["correct"] == is_correct)]
            if group.empty:
                print(f"No {'correct' if is_correct else 'incorrect'} "
                      f"{classes[true_label]} predictions found in this subset - skipping.")
                continue
            samples = group.sample(min(n_per_group, len(group)), random_state=SEED)

            for _, row in samples.iterrows():
                image = Image.open(row["filepath"]).convert("RGB")
                input_tensor = transform(image).unsqueeze(0).to(DEVICE)
                heatmap, explained_class, probabilities = gradcam(input_tensor, None)
                original, heatmap_image, overlay = make_gradcam_overlay(image, heatmap)

                predicted_class = probabilities.argmax(dim=1).item()
                real_prob = probabilities[0, 0].item() * 100
                fake_prob = probabilities[0, 1].item() * 100
                tag = "CORRECT" if is_correct else "INCORRECT"

                fig, axes = plt.subplots(1, 3, figsize=(15, 5))
                axes[0].imshow(original)
                axes[0].set_title("Input image")
                axes[1].imshow(heatmap_image)
                axes[1].set_title(f"Grad-CAM: {classes[explained_class]}")
                axes[2].imshow(overlay)
                axes[2].set_title("Prediction overlay")
                for axis in axes:
                    axis.axis("off")
                fig.suptitle(
                    f"[{tag}] {Path(row['filepath']).name} | "
                    f"Prediction: {classes[predicted_class]} | "
                    f"True: {classes[int(row['label'])]}\n"
                    f"Real: {real_prob:.2f}% | Fake: {fake_prob:.2f}%",
                    fontsize=12,
                )
                plt.tight_layout()
                if save_fig:
                    fig.savefig(
                        OUTPUT_DIR / f"gradcam_{tag.lower()}_{Path(row['filepath']).stem}.png",
                        dpi=150,
                    )
                plt.show()
                plt.close(fig)

    gradcam.close()
    print(f"Grad-CAM figures saved to {OUTPUT_DIR}")


# ---- Execute: explain correct + incorrect Real/Fake predictions ----
run_gradcam_examples(model, test_df, val_transform, n_per_group=2)

## 11. Robustness Evaluation  *(Vishaka)*

Measures the final model's accuracy on corrupted copies of a test subset
(green tint, Gaussian blur, Gaussian noise, JPEG re-compression).

In [ ]:
# ============================================================
# Cell 11 : Robustness Evaluation  (Vishaka)
# ============================================================

def apply_corruption(img, mode):
    """Apply a realistic corruption to a PIL image. (Vishaka)"""
    if mode == "tint":
        arr = np.array(img).astype(np.float32)
        arr[:, :, 1] = np.clip(arr[:, :, 1] * 1.3, 0, 255)  # Green tint
        return Image.fromarray(arr.astype(np.uint8))
    elif mode == "blur":
        return img.filter(ImageFilter.GaussianBlur(radius=2))
    elif mode == "noise":
        arr = np.array(img).astype(np.float32)
        noise = np.random.normal(0, 15, arr.shape)
        return Image.fromarray(np.clip(arr + noise, 0, 255).astype(np.uint8))
    elif mode == "jpeg":
        out = io.BytesIO()
        img.save(out, format="JPEG", quality=30)
        out.seek(0)
        return Image.open(out)
    return img


def run_robustness_eval(model, test_df, transform, n=ROBUSTNESS_SUBSET):
    """Measure model accuracy under corruptions on a test subset. (Vishaka)"""
    print("\n" + "=" * 60)
    print("Robustness Evaluation (corruption accuracy)")
    print("=" * 60)
    model.eval()

    subset = test_df.sample(min(n, len(test_df)), random_state=SEED)
    corruptions = ["original", "tint", "blur", "noise", "jpeg"]
    rows = []

    for mode in corruptions:
        correct = 0
        total = 0
        for _, row in tqdm(subset.iterrows(), total=len(subset),
                           desc=f"Corruption: {mode}"):
            try:
                image = Image.open(row["filepath"]).convert("RGB")
            except Exception:
                continue
            if mode != "original":
                image = apply_corruption(image, mode)
            x = transform(image).unsqueeze(0).to(DEVICE)
            with torch.no_grad():
                with amp_context():
                    pred = model(x).argmax(dim=1).item()
            correct += (pred == int(row["label"]))
            total += 1
        rows.append({"Manipulation": mode.capitalize(),
                     "Accuracy": f"{correct/max(total,1)*100:.2f}%"})

    table = pd.DataFrame(rows)
    print(table.to_string(index=False))
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    table.to_csv(OUTPUT_DIR / "robustness_results.csv", index=False)
    return table


# ---- Execute: corruption robustness table ----
if RUN_ROBUSTNESS_EVAL:
    robustness_table = run_robustness_eval(model, test_df, val_transform)

## 12b. Image Manipulation Testing  *(Aman)*

Applies the **full manipulation set** required for Priority 3 - green tint,
blue tint, brightness, contrast, Gaussian blur, motion blur, JPEG
compression, resize, and crop, plus Gaussian noise - to a held-out test
subset. Shows every manipulation applied to the same source image, then
reports accuracy before vs. after each one.

In [ ]:
# ============================================================
# Cell 12b : Image Manipulation Testing  (Aman)
# ============================================================

from PIL import ImageEnhance

AMAN_MANIPULATIONS = [
    "original", "green_tint", "blue_tint", "brightness", "contrast",
    "gaussian_blur", "motion_blur", "jpeg", "resize", "crop", "noise",
]


def apply_manipulation(img, mode):
    """Apply one manipulation from Aman's Priority 3 list to a PIL image."""
    if mode == "original":
        return img
    if mode == "green_tint":
        arr = np.array(img).astype(np.float32)
        arr[:, :, 1] = np.clip(arr[:, :, 1] * 1.3, 0, 255)
        return Image.fromarray(arr.astype(np.uint8))
    if mode == "blue_tint":
        arr = np.array(img).astype(np.float32)
        arr[:, :, 2] = np.clip(arr[:, :, 2] * 1.3, 0, 255)
        return Image.fromarray(arr.astype(np.uint8))
    if mode == "brightness":
        return ImageEnhance.Brightness(img).enhance(1.4)
    if mode == "contrast":
        return ImageEnhance.Contrast(img).enhance(1.4)
    if mode == "gaussian_blur":
        return img.filter(ImageFilter.GaussianBlur(radius=2))
    if mode == "motion_blur":
        kernel_size = 9
        kernel = np.zeros((kernel_size, kernel_size))
        kernel[kernel_size // 2, :] = np.ones(kernel_size)
        kernel = kernel / kernel_size
        arr = np.array(img).astype(np.float32)
        if CV2_AVAILABLE:
            blurred = cv2.filter2D(arr, -1, kernel)
            return Image.fromarray(np.clip(blurred, 0, 255).astype(np.uint8))
        return img.filter(ImageFilter.GaussianBlur(radius=3))  # fallback, no cv2
    if mode == "jpeg":
        out = io.BytesIO()
        img.save(out, format="JPEG", quality=30)
        out.seek(0)
        return Image.open(out).convert("RGB")
    if mode == "resize":
        w, h = img.size
        small = img.resize((max(1, w // 3), max(1, h // 3)), Image.Resampling.BILINEAR)
        return small.resize((w, h), Image.Resampling.BILINEAR)
    if mode == "crop":
        w, h = img.size
        cw, ch = int(w * 0.7), int(h * 0.7)
        left = (w - cw) // 2
        top = (h - ch) // 2
        return img.crop((left, top, left + cw, top + ch)).resize((w, h), Image.Resampling.LANCZOS)
    if mode == "noise":
        arr = np.array(img).astype(np.float32)
        noise = np.random.normal(0, 15, arr.shape)
        return Image.fromarray(np.clip(arr + noise, 0, 255).astype(np.uint8))
    return img


def show_manipulation_samples(test_df, n_examples=1):
    """Visualize each manipulation applied to the same source image(s). (Aman)"""
    samples = test_df.sample(min(n_examples, len(test_df)), random_state=SEED)
    for _, row in samples.iterrows():
        base_img = Image.open(row["filepath"]).convert("RGB")
        fig, axes = plt.subplots(2, 6, figsize=(20, 7))
        axes = axes.flatten()
        for ax, mode in zip(axes, AMAN_MANIPULATIONS):
            manipulated = apply_manipulation(base_img, mode)
            ax.imshow(manipulated)
            ax.set_title(mode)
            ax.axis("off")
        for ax in axes[len(AMAN_MANIPULATIONS):]:
            ax.axis("off")
        fig.suptitle(f"Manipulation Samples - {Path(row['filepath']).name}", fontsize=13)
        plt.tight_layout()
        os.makedirs(OUTPUT_DIR, exist_ok=True)
        fig.savefig(OUTPUT_DIR / "manipulation_samples.png", dpi=150)
        plt.show()
        plt.close(fig)


def run_manipulation_eval(model, test_df, transform, n=ROBUSTNESS_SUBSET):
    """Before-vs-after accuracy for every Aman manipulation. (Aman)"""
    print("\n" + "=" * 60)
    print("Image Manipulation Testing (Priority 3 - Aman)")
    print("=" * 60)
    model.eval()

    subset = test_df.sample(min(n, len(test_df)), random_state=SEED)
    rows = []
    baseline_acc = None

    for mode in AMAN_MANIPULATIONS:
        correct = 0
        total = 0
        for _, row in tqdm(subset.iterrows(), total=len(subset), desc=f"Manipulation: {mode}"):
            try:
                image = Image.open(row["filepath"]).convert("RGB")
            except Exception:
                continue
            image = apply_manipulation(image, mode)
            x = transform(image).unsqueeze(0).to(DEVICE)
            with torch.no_grad():
                with amp_context():
                    pred = model(x).argmax(dim=1).item()
            correct += (pred == int(row["label"]))
            total += 1
        acc = correct / max(total, 1) * 100
        if mode == "original":
            baseline_acc = acc
        delta = acc - baseline_acc if baseline_acc is not None else 0.0
        rows.append({
            "Manipulation": mode,
            "Accuracy (%)": round(acc, 2),
            "Delta vs Original (pp)": round(delta, 2),
        })

    table = pd.DataFrame(rows)
    print(table.to_string(index=False))
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    table.to_csv(OUTPUT_DIR / "manipulation_results.csv", index=False)
    return table


# ---- Execute: visualize + evaluate every manipulation ----
show_manipulation_samples(test_df, n_examples=1)
manipulation_table = run_manipulation_eval(model, test_df, val_transform)

## 12c. Cross-Domain Testing  *(Raunak)*

Evaluates the trained model specifically on the `nano_banana` rows inside
`test_df` (tagged during `build_splits` in Section 2) - the AI-generated vs.
real images from the "Nano Banana 2.0" dataset, which cover a broader domain
than plain faces. This dataset is intentionally **not held out entirely**:
it is merged into training/val/test like the core face data (see Cell 2),
so the model has actually seen this domain during training. Reporting
`face_main` vs. `nano_banana` accuracy on the test set separately - using
only test-split rows that were never trained on - defines the model's
operational boundary without discarding cross-domain training signal.

In [ ]:
# ============================================================
# Cell 12c : Cross-Domain Testing  (Raunak)
# ============================================================

def show_cross_domain_samples(test_df, n_per_class=3):
    """Visualize sample cross-domain (nano_banana) test images. (Raunak)"""
    cross_df = test_df[test_df["domain"] == "nano_banana"]
    if cross_df.empty:
        print("No nano_banana rows found in test_df - nothing to visualize. "
              "Check USE_EXTRA_DATASET / EXTRA_DATASET in Cell 1.")
        return
    samples = (
        cross_df.groupby("label", group_keys=False)
        .apply(lambda g: g.sample(min(n_per_class, len(g)), random_state=SEED))
        .reset_index(drop=True)
    )
    classes = ["Real", "Fake"]
    n_cols = max(len(samples), 1)
    fig, axes = plt.subplots(1, n_cols, figsize=(4 * n_cols, 4.5))
    if n_cols == 1:
        axes = [axes]
    for i, (_, row) in enumerate(samples.iterrows()):
        img = Image.open(row["filepath"]).convert("RGB")
        axes[i].imshow(img)
        axes[i].set_title(classes[int(row["label"])])
        axes[i].axis("off")
    for i in range(len(samples), n_cols):
        axes[i].axis("off")
    fig.suptitle("Cross-Domain Samples (nano_banana)", fontsize=13)
    plt.tight_layout()
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    fig.savefig(OUTPUT_DIR / "cross_domain_samples.png", dpi=150)
    plt.show()
    plt.close(fig)


def run_cross_domain_eval(model, test_df, transform, criterion):
    """Evaluate accuracy separately per domain to define the model's
    operational boundaries (Priority 4 - Raunak)."""
    print("\n" + "=" * 60)
    print("Cross-Domain Testing (Priority 4 - Raunak)")
    print("=" * 60)

    if "domain" not in test_df.columns:
        print("test_df has no 'domain' column - rebuild splits with the "
              "updated build_splits() first.")
        return None

    results = []
    for domain_name, domain_df in test_df.groupby("domain"):
        loader = DataLoader(
            PreprocessedDataset(domain_df.reset_index(drop=True), transform=transform),
            batch_size=BATCH_SIZE, shuffle=False,
            num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
        )
        _, metrics = validate(model, loader, criterion)
        results.append({
            "Domain": domain_name,
            "N": len(domain_df),
            "Accuracy (%)": round(metrics["acc"] * 100, 2),
            "F1": round(metrics["f1"], 4),
            "AUC": round(metrics["auc"], 4),
        })

    table = pd.DataFrame(results)
    print(table.to_string(index=False))

    if len(table) > 1:
        main_acc = table.loc[table["Domain"] == "face_main", "Accuracy (%)"]
        cross_acc = table.loc[table["Domain"] != "face_main", "Accuracy (%)"]
        if not main_acc.empty and not cross_acc.empty:
            drop = main_acc.values[0] - cross_acc.values.mean()
            print(f"\nOperational boundary: accuracy changes by "
                  f"{drop:.2f} percentage points moving from the core "
                  f"face_main domain to cross-domain (nano_banana) data.")
            print("Interpretation: use this gap as the confidence discount "
                  "to apply when the model is run on images outside the "
                  "core face real-vs-AI domain.")

    os.makedirs(OUTPUT_DIR, exist_ok=True)
    table.to_csv(OUTPUT_DIR / "cross_domain_results.csv", index=False)
    return table


# ---- Execute: visualize + evaluate cross-domain accuracy ----
show_cross_domain_samples(test_df)
cross_domain_table = run_cross_domain_eval(model, test_df, val_transform, criterion)

## 12. Hyperparameter Sweep  *(Rohit)*

Runs the **24 experiments** (learning rate, batch size, weight decay,
dropout, optimizer, scheduler, label smoothing) **directly on this
notebook's actual dataset mix** (`REAL_DIR` + `FAKE_DIR` + `EXTRA_DATASET`)
and logs results to `/kaggle/working/outputs/sweep_comparison.csv`.

> `RUN_SWEEP = True` by default - the sweep now runs fresh on your 3
> datasets instead of reusing values tuned on a different dataset. This
> takes real GPU time (24 experiments x `sweep_epochs` per stage); set to
> `False` to skip and keep whatever is currently in `HPARAMS`. Face
> preprocessing is cached, so each experiment skips RetinaFace.
> After the sweep finishes, the best value per hyperparameter category (by
> Test Accuracy) is printed and written back into `HPARAMS` automatically
> for the rest of this run - copy the printed block into Cell 1 to persist
> it for future runs.

In [ ]:
# ============================================================
# Cell 12 : Hyperparameter Sweep  (Rohit)
# ============================================================

SWEEP_EXPERIMENTS = [
    {"name": "E0_Baseline", "category": "Baseline", "value": "Default",
     "override": {}},
    # 1. Learning Rate
    {"name": "E1a_LR_1e4", "category": "Learning Rate", "value": "1e-4",
     "override": {"learning_rate": 1e-4}},
    {"name": "E1b_LR_5e4", "category": "Learning Rate", "value": "5e-4",
     "override": {"learning_rate": 5e-4}},
    {"name": "E1c_LR_1e3", "category": "Learning Rate", "value": "1e-3",
     "override": {"learning_rate": 1e-3}},
    {"name": "E1d_LR_5e3", "category": "Learning Rate", "value": "5e-3",
     "override": {"learning_rate": 5e-3}},
    # 2. Batch Size
    {"name": "E2a_BS_32", "category": "Batch Size", "value": "32",
     "override": {"batch_size": 32}},
    {"name": "E2b_BS_64", "category": "Batch Size", "value": "64",
     "override": {"batch_size": 64}},
    {"name": "E2c_BS_128", "category": "Batch Size", "value": "128",
     "override": {"batch_size": 128}},
    # 3. Weight Decay
    {"name": "E3a_WD_0.0", "category": "Weight Decay", "value": "0.0",
     "override": {"weight_decay": 0.0}},
    {"name": "E3b_WD_0.01", "category": "Weight Decay", "value": "0.01",
     "override": {"weight_decay": 0.01}},
    {"name": "E3c_WD_0.05", "category": "Weight Decay", "value": "0.05",
     "override": {"weight_decay": 0.05}},
    {"name": "E3d_WD_0.1", "category": "Weight Decay", "value": "0.1",
     "override": {"weight_decay": 0.1}},
    # 4. Dropout Rate
    {"name": "E4a_Drop_0.0", "category": "Dropout Rate", "value": "0.0",
     "override": {"dropout": 0.0}},
    {"name": "E4b_Drop_0.2", "category": "Dropout Rate", "value": "0.2",
     "override": {"dropout": 0.2}},
    {"name": "E4c_Drop_0.3", "category": "Dropout Rate", "value": "0.3",
     "override": {"dropout": 0.3}},
    {"name": "E4d_Drop_0.5", "category": "Dropout Rate", "value": "0.5",
     "override": {"dropout": 0.5}},
    # 5. Optimizer
    {"name": "E5a_Opt_Adam", "category": "Optimizer", "value": "Adam",
     "override": {"optimizer": "adam"}},
    {"name": "E5b_Opt_AdamW", "category": "Optimizer", "value": "AdamW",
     "override": {"optimizer": "adamw"}},
    # 6. Scheduler
    {"name": "E6a_Sched_Cosine", "category": "Scheduler", "value": "Cosine",
     "override": {"scheduler": "cosine"}},
    {"name": "E6b_Sched_Plateau", "category": "Scheduler", "value": "Plateau",
     "override": {"scheduler": "plateau"}},
    # 7. Label Smoothing
    {"name": "E7a_LS_0.0", "category": "Label Smoothing", "value": "0.0",
     "override": {"label_smoothing": 0.0}},
    {"name": "E7b_LS_0.05", "category": "Label Smoothing", "value": "0.05",
     "override": {"label_smoothing": 0.05}},
    {"name": "E7c_LS_0.10", "category": "Label Smoothing", "value": "0.10",
     "override": {"label_smoothing": 0.10}},
]


def run_sweep(sweep_epochs=3):
    """Run every hyperparameter experiment on THIS dataset mix and log
    results. (Rohit)

    Previously HPARAMS reused values tuned on a different dataset; this
    function always rebuilds splits from the CURRENT REAL_DIR / FAKE_DIR /
    EXTRA_DATASET globals, so results reflect the actual 3-dataset mix in
    use right now.
    """
    print("=" * 60)
    print("Hyperparameter Sweep (fresh run on current dataset mix)")
    print("=" * 60)

    records = []
    for exp in SWEEP_EXPERIMENTS:
        hparams = {**HPARAMS, **exp["override"]}

        random.seed(SEED)
        np.random.seed(SEED)
        torch.manual_seed(SEED)

        print("\n" + "=" * 60)
        print(f"Experiment : {exp['name']}")
        print("=" * 60)

        # Rebuild data (preprocessing is cached, so this is fast)
        tr_df, va_df, te_df = build_splits(
            REAL_DIR, FAKE_DIR,
            extra_dataset=EXTRA_DATASET if USE_EXTRA_DATASET else None,
        )
        if DO_FACE_PREPROCESSING:
            tr_df, va_df, te_df = run_face_preprocessing(
                tr_df, va_df, te_df, PREP_DIR
            )

        t_loader, v_loader, e_loader = build_dataloaders(
            tr_df, va_df, te_df,
            batch_size=hparams["batch_size"],
            robust_aug=ROBUST_TRAIN_AUG,
        )

        m = build_model(dropout=hparams["dropout"]).to(DEVICE)
        crit = nn.CrossEntropyLoss(label_smoothing=hparams["label_smoothing"])

        t0 = time.time()

        # Stage 1
        opt = build_optimizer(m, hparams["optimizer"], hparams["learning_rate"],
                              hparams["weight_decay"], stage=1)
        sch = build_scheduler(opt, hparams["scheduler"], sweep_epochs)
        va_acc = train_stage(
            m, (t_loader, v_loader, e_loader), opt, sch, crit, sweep_epochs,
            f"Stage 1 (exp {exp['name']})", hparams["scheduler"],
        )

        # Stage 2
        ckpt = torch.load(MODEL_SAVE_PATH, map_location=DEVICE)
        m.load_state_dict(ckpt["model_state_dict"])
        va_acc = max(va_acc, ckpt["val_acc"])

        blocks = list(m.features.children())
        start = int(len(blocks) * 0.75)
        for b in blocks[start:]:
            for prm in b.parameters():
                prm.requires_grad = True

        opt = build_optimizer(m, hparams["optimizer"], hparams["lr_stage2"],
                              hparams["weight_decay"], stage=2)
        sch = build_scheduler(opt, hparams["scheduler"], sweep_epochs)
        va_acc = train_stage(
            m, (t_loader, v_loader, e_loader), opt, sch, crit, sweep_epochs,
            f"Stage 2 (exp {exp['name']})", hparams["scheduler"],
        )

        # Evaluate
        m.load_state_dict(
            torch.load(MODEL_SAVE_PATH, map_location=DEVICE)["model_state_dict"]
        )
        test_loss, test_metrics = evaluate(m, e_loader, crit, save_fig=False)

        records.append({
            "Experiment": exp["name"],
            "Hyperparameter": exp["category"],
            "Value": exp["value"],
            "Val Acc (%)": round(va_acc * 100, 2),
            "Test Acc (%)": round(test_metrics["acc"] * 100, 2),
            "Test Precision": round(test_metrics["precision"], 4),
            "Test Recall": round(test_metrics["recall"], 4),
            "Test F1": round(test_metrics["f1"], 4),
            "Test AUC": round(test_metrics["auc"], 4),
            "Test Loss": round(test_loss, 4),
            "Time (s)": round(time.time() - t0, 1),
        })

    df = pd.DataFrame(records)
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    csv_path = OUTPUT_DIR / "sweep_comparison.csv"
    df.to_csv(csv_path, index=False)

    print("\n=== HYPERPARAMETER SWEEP RESULTS ===")
    print(df.to_string(index=False))
    print(f"\nComparison table saved to {csv_path}")

    best_row = df.loc[df["Test Acc (%)"].idxmax()]
    print("\n" + "=" * 60)
    print(f"Best Experiment : {best_row['Experiment']} "
          f"({best_row['Hyperparameter']} = {best_row['Value']})")
    print(f"Test Acc (%)    : {best_row['Test Acc (%)']}")
    print("=" * 60)

    return df, best_row


def apply_best_hparams(sweep_df):
    """Pick the best value per hyperparameter category (by Test Acc) and
    fold it into the global HPARAMS. Prints a copy-paste-ready block for
    persisting the change into Cell 1."""
    param_key_map = {
        "Learning Rate": "learning_rate",
        "Batch Size": "batch_size",
        "Weight Decay": "weight_decay",
        "Dropout Rate": "dropout",
        "Optimizer": "optimizer",
        "Scheduler": "scheduler",
        "Label Smoothing": "label_smoothing",
    }

    print("\nRecommended HPARAMS (best value per category, by Test Acc):")
    for category, key in param_key_map.items():
        cat_rows = sweep_df[sweep_df["Hyperparameter"] == category]
        if cat_rows.empty:
            continue
        best_cat_row = cat_rows.loc[cat_rows["Test Acc (%)"].idxmax()]
        matching_exp = next(
            e for e in SWEEP_EXPERIMENTS
            if e["value"] == best_cat_row["Value"] and key in e["override"]
        )
        typed_value = matching_exp["override"][key]
        HPARAMS[key] = typed_value
        print(f"  {key:<15} -> {typed_value!r}  (from {best_cat_row['Experiment']}, "
              f"{best_cat_row['Test Acc (%)']}% test acc)")

    print("\nHPARAMS updated in-memory for the rest of this run:")
    print(HPARAMS)
    print("\nCopy this dict into Cell 1 (Imports & Configuration) to persist it "
          "across future runs.")


# ---------------------------------------------------------------
# Execute: run the sweep fresh on the current 3-dataset mix, then
# auto-adopt the winning hyperparameter values.
# ---------------------------------------------------------------
RUN_SWEEP = True   # set False to skip and keep the current HPARAMS as-is

if RUN_SWEEP:
    sweep_df, best_overall = run_sweep(sweep_epochs=3)
    apply_best_hparams(sweep_df)

## 12d. Retrain With Best Hyperparameters  *(Rohit)*

The sweep above (Section 12) picks the best value per hyperparameter
category but the *original* model (Sections 8-9, trained with the old
placeholder `HPARAMS`) is never retrained with them. This section retrains
one more full model using the sweep's winning `HPARAMS` and saves it
separately (`mobilenetv3_tuned.pth`), so Priority 2's upload-test section at
the end of the notebook can show a real side-by-side: baseline HPARAMS vs.
swept/tuned HPARAMS, on the same image.

> `RETRAIN_WITH_BEST_HPARAMS = True` by default - runs only if `RUN_SWEEP`
> above actually executed (otherwise there is nothing new to retrain with).
> This is a third full Stage 1 + Stage 2 training pass.

In [ ]:
# ============================================================
# Cell 12d : Retrain With Best Hyperparameters  (Rohit)
# ============================================================

MODEL_SAVE_PATH_TUNED = WORK_DIR / "mobilenetv3_tuned.pth"
RETRAIN_WITH_BEST_HPARAMS = True   # set False to skip the tuned-HPARAMS retrain

model_tuned = None
if RETRAIN_WITH_BEST_HPARAMS and RUN_SWEEP:
    model_tuned = train_full_model(
        robust_aug=ROBUST_TRAIN_AUG,
        save_path=MODEL_SAVE_PATH_TUNED,
        run_label="Tuned-HPARAMS Model",
    )
    _, tuned_metrics = validate(model_tuned, test_loader, criterion)
    _, baseline_metrics = validate(model, test_loader, criterion)
    hparam_compare = pd.DataFrame([
        {"Model": "Baseline HPARAMS", "Test Acc (%)": round(baseline_metrics["acc"] * 100, 2),
         "F1": round(baseline_metrics["f1"], 4), "AUC": round(baseline_metrics["auc"], 4)},
        {"Model": "Swept/Tuned HPARAMS", "Test Acc (%)": round(tuned_metrics["acc"] * 100, 2),
         "F1": round(tuned_metrics["f1"], 4), "AUC": round(tuned_metrics["auc"], 4)},
    ])
    print("\n" + "=" * 60)
    print("Baseline vs. Tuned HPARAMS - Test Set Comparison")
    print("=" * 60)
    print(hparam_compare.to_string(index=False))
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    hparam_compare.to_csv(OUTPUT_DIR / "hparam_baseline_vs_tuned.csv", index=False)
else:
    print("RETRAIN_WITH_BEST_HPARAMS is False (or RUN_SWEEP did not run) - "
          "skipping tuned-HPARAMS retrain. model_tuned will be unavailable "
          "in the Priority 2 upload-test section below.")

## 14. Upload & Test Your Own Images - Priority 1-6  *(All)*

Every section below lets you upload your own image(s) and immediately see
how a specific piece of this project behaves on unseen input, matching the
6 testing priorities from the task list. Run this helper cell once first -
every priority section below depends on it.

In [ ]:
# ============================================================
# Shared inference helpers used by every Priority 1-6 upload section below
# ============================================================
import ipywidgets as widgets

CLASSES = ["Real", "Fake"]


def _read_uploaded_image(uploader):
    """Return (filename, PIL.Image) for the first uploaded file, or None
    if nothing has been uploaded to this widget yet."""
    if uploader is None or len(uploader.value) == 0:
        return None
    files = (
        uploader.value if isinstance(uploader.value, dict)
        else {f["name"]: f for f in uploader.value}
    )
    name, info = next(iter(files.items()))
    content = info["content"] if isinstance(info, dict) and "content" in info else None
    if content is None:
        return None
    return name, Image.open(io.BytesIO(content)).convert("RGB")


def predict_with_model(model_obj, image, transform):
    """Run one model on one PIL image -> (pred_class_idx, real_pct, fake_pct)."""
    model_obj.eval()
    x = transform(image).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        with amp_context():
            probs = torch.softmax(model_obj(x), dim=1)[0].cpu()
    pred = int(probs.argmax())
    return pred, probs[0].item() * 100, probs[1].item() * 100


def gradcam_overlay_for(model_obj, image, transform, alpha=0.45):
    """Run Grad-CAM on one model/image ->
    (original, heatmap_img, overlay, pred, real_pct, fake_pct)."""
    gc_ = GradCAM(model_obj, model_obj.features[-1])
    x = transform(image).unsqueeze(0).to(DEVICE)
    heatmap, _explained_class, probabilities = gc_(x, None)
    gc_.close()
    display_img = image.convert("RGB").resize((IMG_SIZE, IMG_SIZE))
    heatmap_rgb = (plt.get_cmap("jet")(heatmap)[..., :3] * 255).astype(np.uint8)
    heatmap_img = Image.fromarray(heatmap_rgb).resize(display_img.size)
    overlay = Image.blend(display_img, heatmap_img, alpha=alpha)
    pred = int(probabilities.argmax(dim=1))
    real_pct = probabilities[0, 0].item() * 100
    fake_pct = probabilities[0, 1].item() * 100
    return display_img, heatmap_img, overlay, pred, real_pct, fake_pct


print("Shared inference helpers ready - run each Priority section's upload "
      "cell, then its follow-up cell.")

## 15. Priority 1 - Robust Augmentation (With vs. Without)  *(Vishakha)*

Upload a face image. It is run through **both** the "with augmentation"
model (`model`) and the "no-augmentation" model (`model_noaug`, Section 9b)
so you can see directly whether training with augmentation changes the
prediction/confidence on your own image.

In [ ]:
uploader_p1 = widgets.FileUpload(accept="image/*", multiple=False)
display(uploader_p1)
print("Upload one image above, then run the next cell.")

In [ ]:
result = _read_uploaded_image(uploader_p1)
if result is None:
    print("No image uploaded yet - use the cell above first.")
elif "model_noaug" not in dir() or model_noaug is None:
    print("model_noaug is not available - set TRAIN_NOAUG_MODEL = True in "
          "Section 9b and re-run that section first.")
else:
    name, image = result
    pred_a, real_a, fake_a = predict_with_model(model, image, val_transform)
    pred_b, real_b, fake_b = predict_with_model(model_noaug, image, val_transform)

    compare_table = pd.DataFrame([
        {"Model": "With Augmentation", "Prediction": CLASSES[pred_a],
         "Real %": round(real_a, 2), "Fake %": round(fake_a, 2)},
        {"Model": "Without Augmentation", "Prediction": CLASSES[pred_b],
         "Real %": round(real_b, 2), "Fake %": round(fake_b, 2)},
    ])
    print(f"Image: {name}")
    print(compare_table.to_string(index=False))

    fig, ax = plt.subplots(1, 1, figsize=(4, 4))
    ax.imshow(image)
    ax.set_title("Uploaded Image")
    ax.axis("off")
    plt.show()
    plt.close(fig)

## 16. Priority 2 - Hyperparameter Optimization (Baseline vs. Tuned)  *(Rohit)*

Upload a face image to compare the original placeholder-`HPARAMS` model
(`model`) against the model retrained with the sweep's best config
(`model_tuned`, Section 12d).

In [ ]:
uploader_p2 = widgets.FileUpload(accept="image/*", multiple=False)
display(uploader_p2)
print("Upload one image above, then run the next cell.")

In [ ]:
result = _read_uploaded_image(uploader_p2)
if result is None:
    print("No image uploaded yet - use the cell above first.")
elif "model_tuned" not in dir() or model_tuned is None:
    print("model_tuned is not available - set RUN_SWEEP = True (Section 12) "
          "and RETRAIN_WITH_BEST_HPARAMS = True (Section 12d), then re-run "
          "those sections first.")
else:
    name, image = result
    pred_a, real_a, fake_a = predict_with_model(model, image, val_transform)
    pred_b, real_b, fake_b = predict_with_model(model_tuned, image, val_transform)

    compare_table = pd.DataFrame([
        {"Model": "Baseline HPARAMS", "Prediction": CLASSES[pred_a],
         "Real %": round(real_a, 2), "Fake %": round(fake_a, 2)},
        {"Model": "Swept/Tuned HPARAMS", "Prediction": CLASSES[pred_b],
         "Real %": round(real_b, 2), "Fake %": round(fake_b, 2)},
    ])
    print(f"Image: {name}")
    print(compare_table.to_string(index=False))

    fig, ax = plt.subplots(1, 1, figsize=(4, 4))
    ax.imshow(image)
    ax.set_title("Uploaded Image")
    ax.axis("off")
    plt.show()
    plt.close(fig)

## 17. Priority 3 - Image Manipulation Robustness  *(Aman)*

Upload a face image. It is run through every manipulation from Section 12b
(green/blue tint, brightness, contrast, Gaussian blur, motion blur, JPEG,
resize, crop, noise) and the model's prediction is reported for each.

In [ ]:
uploader_p3 = widgets.FileUpload(accept="image/*", multiple=False)
display(uploader_p3)
print("Upload one image above, then run the next cell.")

In [ ]:
result = _read_uploaded_image(uploader_p3)
if result is None:
    print("No image uploaded yet - use the cell above first.")
else:
    name, image = result
    rows = []
    fig, axes = plt.subplots(2, 6, figsize=(20, 7))
    axes = axes.flatten()
    for ax, mode in zip(axes, AMAN_MANIPULATIONS):
        manipulated = apply_manipulation(image, mode)
        pred, real_pct, fake_pct = predict_with_model(model, manipulated, val_transform)
        rows.append({"Manipulation": mode, "Prediction": CLASSES[pred],
                     "Real %": round(real_pct, 2), "Fake %": round(fake_pct, 2)})
        ax.imshow(manipulated)
        ax.set_title(f"{mode}\n{CLASSES[pred]}", fontsize=9)
        ax.axis("off")
    for ax in axes[len(AMAN_MANIPULATIONS):]:
        ax.axis("off")
    fig.suptitle(f"Manipulation Predictions - {name}", fontsize=13)
    plt.tight_layout()
    plt.show()
    plt.close(fig)

    print(pd.DataFrame(rows).to_string(index=False))

## 18. Priority 4 - Cross-Domain Generalization  *(Raunak)*

Upload an image OUTSIDE the core face domain (an animal, a landscape, an
object - AI-generated or real) to see how the model behaves on it. Section
12c already reports this quantitatively on the full test set; use this cell
to sanity-check on your own picture. Treat the result with the
operational-boundary gap from Section 12c in mind, not as a fully reliable
verdict outside the face domain.

In [ ]:
uploader_p4 = widgets.FileUpload(accept="image/*", multiple=False)
display(uploader_p4)
print("Upload one (ideally non-face) image above, then run the next cell.")

In [ ]:
result = _read_uploaded_image(uploader_p4)
if result is None:
    print("No image uploaded yet - use the cell above first.")
else:
    name, image = result
    pred, real_pct, fake_pct = predict_with_model(model, image, val_transform)

    print(f"Image: {name}")
    print(f"Prediction : {CLASSES[pred]}")
    print(f"Real: {real_pct:.2f}% | Fake: {fake_pct:.2f}%")
    print("\nReminder: see Section 12c for the measured face_main vs. "
          "nano_banana accuracy gap - treat this prediction with reduced "
          "confidence if your image is far outside the face domain.")

    fig, ax = plt.subplots(1, 1, figsize=(4, 4))
    ax.imshow(image)
    ax.set_title(f"{CLASSES[pred]} ({max(real_pct, fake_pct):.1f}%)")
    ax.axis("off")
    plt.show()
    plt.close(fig)

## 19. Priority 5 - Explainability (Grad-CAM)  *(Somendu)*

Upload any image to see the model's prediction alongside a Grad-CAM
heatmap showing which regions drove that decision.

In [ ]:
uploader_p5 = widgets.FileUpload(accept="image/*", multiple=False)
display(uploader_p5)
print("Upload one image above, then run the next cell.")

In [ ]:
result = _read_uploaded_image(uploader_p5)
if result is None:
    print("No image uploaded yet - use the cell above first.")
else:
    name, image = result
    original, heatmap_img, overlay, pred, real_pct, fake_pct = gradcam_overlay_for(
        model, image, val_transform
    )

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(original)
    axes[0].set_title("Input Image")
    axes[1].imshow(heatmap_img)
    axes[1].set_title(f"Grad-CAM ({CLASSES[pred]})")
    axes[2].imshow(overlay)
    axes[2].set_title("Overlay")
    for ax in axes:
        ax.axis("off")
    fig.suptitle(f"{name}\nPrediction: {CLASSES[pred]} | "
                f"Real: {real_pct:.2f}% | Fake: {fake_pct:.2f}%", fontsize=12)
    plt.tight_layout()
    plt.show()
    plt.close(fig)

## 20. Priority 6 - End-to-End Inference Pipeline  *(Vishakha)*

Upload one or more images and get a prediction with confidence scores and
Grad-CAM - the full inference pipeline a real user would experience,
end-to-end.

In [ ]:
# ============================================================
# Cell 13a : Upload widget
# ============================================================
import ipywidgets as widgets

uploader = widgets.FileUpload(
    accept="image/*",
    multiple=True
)

display(uploader)
print("Upload images above, then run the next cell.")

In [ ]:
# ============================================================
# Cell 13b : Predict uploaded images + Grad-CAM
# ============================================================

classes = ["Real", "Fake"]

if "uploader" not in dir() or len(uploader.value) == 0:
    print("No images uploaded yet - use the cell above first.")
else:
    gradcam = GradCAM(model, model.features[-1])

    def make_gradcam_overlay(image, heatmap, alpha=0.45):
        image = image.convert("RGB").resize((IMG_SIZE, IMG_SIZE))

        heatmap_rgb = (
            plt.get_cmap("jet")(heatmap)[..., :3] * 255
        ).astype(np.uint8)

        heatmap_image = Image.fromarray(heatmap_rgb).resize(image.size)
        overlay = Image.blend(image, heatmap_image, alpha=alpha)

        return image, heatmap_image, overlay

    files = (
        uploader.value
        if isinstance(uploader.value, dict)
        else {f["name"]: f for f in uploader.value}
    )

    for name, info in files.items():

        content = (
            info["content"]
            if isinstance(info, dict) and "content" in info
            else None
        )

        if content is None:
            continue

        image = Image.open(io.BytesIO(content)).convert("RGB")

        x = val_transform(image).unsqueeze(0).to(DEVICE)

        # Grad-CAM (also returns prediction probabilities)
        heatmap, explained_class, probabilities = gradcam(x)

        pred = probabilities.argmax(dim=1).item()
        real_p = probabilities[0, 0].item() * 100
        fake_p = probabilities[0, 1].item() * 100

        print("=" * 60)
        print(name)
        print(f"Prediction : {classes[pred]}")
        print(f"Real: {real_p:.2f}% | Fake: {fake_p:.2f}%")
        print("=" * 60)

        original, heatmap_img, overlay = make_gradcam_overlay(image, heatmap)

        fig, axes = plt.subplots(1, 3, figsize=(15, 5))

        axes[0].imshow(original)
        axes[0].set_title("Input Image")

        axes[1].imshow(heatmap_img)
        axes[1].set_title(f"Grad-CAM ({classes[explained_class]})")

        axes[2].imshow(overlay)
        axes[2].set_title("Overlay")

        for ax in axes:
            ax.axis("off")

        fig.suptitle(
            f"{name}\n"
            f"Prediction: {classes[pred]} | "
            f"Real: {real_p:.2f}% | Fake: {fake_p:.2f}%",
            fontsize=12,
        )

        plt.tight_layout()
        plt.show()

    gradcam.close()